# ممیزی snapshot زندهٔ Divar × Laya

## tl;dr

این دفترچه فقط تا `outputBytes` ثبت‌شده در manifest می‌خواند؛ بنابراین اجرای هم‌زمان batch باعث خواندن ردیف نیمه‌نوشته یا جلوتر از checkpoint نمی‌شود. خروجی‌ها proposalهای فرضی مشتق‌شده از آگهی عرضه‌اند، نه نیاز واقعی یا برچسب gold.

## Context & Methods

واحد منبع: یک آگهی عرضهٔ ملکی نرمال‌شده. واحد خروجی: یک متن فرضی و تصمیم‌های Laya برای همان آگهی. فقط aggregateها چاپ می‌شوند؛ متن آگهی خام نمایش داده نمی‌شود.

## Data

ورودی‌ها: `data/divar/divar-property-offer-facts-v2.jsonl` و manifest آن، به‌علاوهٔ `data/divar/divar-hypothetical-needs-laya-full-v4-2026-09-26.jsonl` و manifest زندهٔ آن. مدل: `convaiinnovations/laya-multilingual` محلی.

## Results

سلول‌های زیر پرچم‌های synthetic/training، یکتایی، پوشش شهر/محله/دسته، و توافق خروجی Laya با targetهای فرضی را تا checkpoint ثبت‌شده محاسبه می‌کنند. توافق با target منبعی، دقت نیاز کاربران نیست؛ گزینه‌های دسته‌بندی این نسخه نیز با دستهٔ هدف منبع محدود شده‌اند و معیار دسته‌بندی از نشت برچسب رنج می‌برد.

## Takeaways

این artifact فقط برای shadow QA و آزمون pipeline است. آن را به مدل آموزشی یا آمار تقاضای واقعی وارد نکنید. مجوز dataset در manifest منبع ODbL است، اما بررسی حقوق محتوای آگهی و مجوز بازتوزیع جداگانه و همچنان باز است.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
SOURCE = ROOT / 'data/divar/divar-property-offer-facts-v2.jsonl'
OUTPUT = ROOT / 'data/divar/divar-hypothetical-needs-laya-full-v4-2026-09-26.jsonl'
source_manifest = json.loads(SOURCE.with_suffix(SOURCE.suffix + '.manifest.json').read_text(encoding='utf-8'))
run_manifest_path = OUTPUT.with_suffix(OUTPUT.suffix + '.manifest.json')
run_manifest = json.loads(run_manifest_path.read_text(encoding='utf-8'))
assert source_manifest['dataset'] == 'divarofficial/real_estate_ads'
assert run_manifest['model'] == 'convaiinnovations/laya-multilingual'
assert run_manifest['taskType'] == 'divar-counterfactual-post-need-laya-proposal/v1'
assert run_manifest['status'] in {'running', 'complete'}
assert OUTPUT.stat().st_size >= run_manifest['outputBytes']
print(json.dumps({
    'source_rows': source_manifest['outputRows'],
    'source_unique_text_groups': source_manifest['normalizedTextGroups'],
    'source_sha256': run_manifest['sourceCorpusSha256'],
    'model': run_manifest['model'],
    'run_status': run_manifest['status'],
    'committed_rows': run_manifest['rowsProcessed'],
    'committed_bytes': run_manifest['outputBytes'],
    'last_source_row_ordinal': run_manifest['lastSourceRowOrdinal'],
    'updated_at_utc': run_manifest['updatedAt'],
}, ensure_ascii=False, indent=2))

In [ ]:
from collections import Counter

FIELDS = ('category_candidate', 'property_kind', 'transaction_type', 'deed_type', 'usage', 'parking', 'elevator', 'storage')
profile = {
    'rows': 0, 'invalid_synthetic_flags': 0, 'duplicate_source_ids': 0,
    'duplicate_text_groups': 0, 'mapped_city_rows': 0, 'exact_neighborhood_rows': 0,
    'source_ids': set(), 'text_groups': set(), 'cities': set(), 'categories': set(),
    'city_neighborhood_pairs': set(),
    'field_metrics': {key: {'answers': 0, 'agreement': 0, 'unknown_targets': 0, 'false_proposals': 0} for key in FIELDS},
}

def inspect_row(row):
    profile['rows'] += 1
    if row.get('synthetic') is not True or row.get('realNeedGroundTruth') is not False or row.get('trainingEligible') is not False:
        profile['invalid_synthetic_flags'] += 1
    source = row.get('source') or {}
    source_id = source.get('sourceExampleId')
    group = source.get('normalizedTextGroupSha256')
    if source_id in profile['source_ids']:
        profile['duplicate_source_ids'] += 1
    profile['source_ids'].add(source_id)
    if group in profile['text_groups']:
        profile['duplicate_text_groups'] += 1
    profile['text_groups'].add(group)
    hypothetical = row['hypotheticalNeed']
    location = hypothetical.get('sourceOfferLocation') or {}
    city = location.get('appCitySlug')
    if city:
        profile['mapped_city_rows'] += 1
        profile['cities'].add(city)
    if location.get('neighborhoodMatch') == 'exact_official_crosswalk' and location.get('appNeighborhoodId'):
        profile['exact_neighborhood_rows'] += 1
        profile['city_neighborhood_pairs'].add((city, location['appNeighborhoodId']))
    targets = hypothetical.get('targetDecisions') or {}
    category = (targets.get('category_candidate') or {}).get('value')
    if category:
        profile['categories'].add(category)
    answers = (row.get('laya') or {}).get('answers') or {}
    for key in FIELDS:
        expected = str((targets.get(key) or {}).get('value', 'unknown'))
        answer = answers.get(key) or {}
        predicted = str(answer.get('choice', answer.get('noul', answer.get('value', 'unknown'))))
        metric = profile['field_metrics'][key]
        metric['answers'] += key in answers
        metric['agreement'] += predicted == expected
        if expected == 'unknown':
            metric['unknown_targets'] += 1
            metric['false_proposals'] += predicted != 'unknown'

committed_bytes = run_manifest['outputBytes']
remaining = committed_bytes
buffer = b''
with OUTPUT.open('rb') as stream:
    while remaining:
        chunk = stream.read(min(1024 * 1024, remaining))
        if not chunk:
            raise EOFError('Output ended before the manifest checkpoint.')
        remaining -= len(chunk)
        lines = (buffer + chunk).split(b'\n')
        buffer = lines.pop()
        for raw_line in lines:
            if raw_line:
                inspect_row(json.loads(raw_line))
if buffer:
    raise ValueError('The committed checkpoint ends inside a JSONL row.')
assert profile['rows'] == run_manifest['rowsProcessed']

def pct(numerator, denominator):
    return round(100 * numerator / denominator, 2) if denominator else None

report = {
    'rows': profile['rows'],
    'invalid_synthetic_flags': profile['invalid_synthetic_flags'],
    'duplicate_source_ids': profile['duplicate_source_ids'],
    'duplicate_text_groups': profile['duplicate_text_groups'],
    'mapped_city_rows_pct': pct(profile['mapped_city_rows'], profile['rows']),
    'mapped_city_slugs': len(profile['cities']),
    'exact_neighborhood_rows_pct': pct(profile['exact_neighborhood_rows'], profile['rows']),
    'exact_city_neighborhood_pairs': len(profile['city_neighborhood_pairs']),
    'category_leaves': sorted(profile['categories']),
    'model_proxy_metrics_not_ground_truth': {
        key: {
            'answer_coverage_pct': pct(value['answers'], profile['rows']),
            'agreement_pct': pct(value['agreement'], profile['rows']),
            'false_proposal_among_unknown_targets_pct': pct(value['false_proposals'], value['unknown_targets']),
        }
        for key, value in profile['field_metrics'].items()
    },
}
print(json.dumps(report, ensure_ascii=False, indent=2))

## Interpretation

Agreement percentages compare Laya with labels derived from seller/agent-side source facts and from a deterministic counterfactual template. They do not estimate accuracy on real seekers. In this version, category options are restricted using the expected source category, so category agreement is target-conditioned and invalid as an independent score. The source itself has incomplete app-city/category/neighborhood coverage; missing regions must remain missing rather than being fabricated and counted as observed needs.